In [1]:
import pandas as pd
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
import string
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout
from tensorflow.keras.utils import to_categorical
from random import randint

In [2]:
df = pd.read_csv("output/raw.csv")

In [34]:
# text = ''.join([c for c in ' '.join(df.dropna().values.flatten()).lower() if c not in string.punctuation])
text = ' '.join([sentence.strip() if sentence.strip()[-1] == '.' else sentence.strip() + '.' for sentence in df.dropna().values.flatten()]).lower()
words = word_tokenize(text)
n_words = len(words)
unique_words = len(set(words))

print('Total Words: %d' % n_words)
print('Unique Words: %d' % unique_words)

Total Words: 65523
Unique Words: 8341


In [21]:
unique_chars = sorted(list(set(text)))
char_to_index = {char: idx for idx, char in enumerate(unique_chars)}
char_to_index

{' ': 0,
 '.': 1,
 'a': 2,
 'b': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'l': 11,
 'm': 12,
 'n': 13,
 'o': 14,
 'p': 15,
 'r': 16,
 's': 17,
 't': 18,
 'u': 19,
 'w': 20,
 'y': 21}

In [22]:
input_sequence = []
output_words = []
input_seq_length = 40

for i in range(0, len(text) - input_seq_length , 1):
    in_seq = text[i:i + input_seq_length]
    out_seq = text[i + input_seq_length]
    input_sequence.append([char_to_index[word] for word in in_seq])
    output_words.append(char_to_index[out_seq])

In [23]:
index_to_char = {idx: char for char, idx in char_to_index.items()}

In [24]:
X = np.reshape(input_sequence, (len(input_sequence), input_seq_length, 1))
X = X / float(len(unique_chars))

y = to_categorical(output_words, num_classes=len(unique_chars))

In [25]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (69, 40, 1)
y shape: (69, 22)


In [26]:
model = Sequential([
    LSTM(128, input_shape=(input_seq_length, 1), return_sequences=True),
    LSTM(128),
    Dense(len(unique_chars), activation='softmax')
])
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam')

C:\Users\lewjj\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                        │ (None, 40, 128)             │          66,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 22)                  │           2,838 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 200,982 (785.09 KB)

 Trainable params: 200,982 (785.09 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.fit(X, y, batch_size=128, epochs=10, verbose=1)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 3.0864
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 3.0651
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 3.0408
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 3.0077
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 2.9574
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 2.8822
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 2.8065
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 2.8308
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - loss: 2.8216
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - loss: 2.7859


In [31]:
text[start_index:start_index + input_seq_length]

'gins with a single step. happiness is a '

In [33]:
import random

# Pick a random starting sequence
start_index = random.randint(0, len(text) - input_seq_length - 1)
seed_text = text[start_index:start_index + input_seq_length]

# Generate characters
generated_text = seed_text
for _ in range(200):  # Generate 200 characters
    input_seq = np.array([[char_to_index[char] for char in seed_text]]).reshape(1, input_seq_length, 1)
    input_seq = input_seq / float(len(unique_chars))

    # Predict the next character
    predicted_index = np.argmax(model.predict(input_seq, verbose=0))
    next_char = unique_chars[predicted_index]

    # Append to generated text and update seed
    generated_text += next_char
    seed_text = seed_text[1:] + next_char

print(generated_text)

es begins with a single step. happiness                                                                                                                                                                                                         


In [38]:
from transformers import pipeline

# Load a pre-trained model for text classification
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# Define the sentence and candidate labels (themes)
sentence = "The stock market crashed due to unexpected economic downturn."
candidate_labels = ["finance", "sports", "politics", "technology", "health"]

# Perform the classification
result = classifier(sentence, candidate_labels)

# Print the result
print(f"Sentence: {sentence}")
print(f"Detected Theme: {result['labels'][0]} with confidence {result['scores'][0]:.2f}")

Device set to use cpu


Sentence: The stock market crashed due to unexpected economic downturn.
Detected Theme: finance with confidence 0.95
